In [1]:
from __future__ import annotations

import os
import re
import shutil
import subprocess
import time
from difflib import unified_diff
from pathlib import Path
from typing import Dict, List, Optional, Tuple


# 本 notebook 不执行 CMake 配置或构建，只测试现有 build-riscv-release/compiler。
# ==================== 可直接修改的测试配置 ====================
MAX_CASES: Optional[int] = None       # 例如 5；None 表示运行全部样例
SINGLE_CASES: Tuple[str, ...] = ()    # 例如 ("functional/00_main.sy",)
STOP_ON_FIRST_FAIL = False

COMPILE_TIMEOUT = 500
ASSEMBLE_TIMEOUT = 60
LINK_TIMEOUT = 60
RUN_TIMEOUT = 300

RISCV_MARCH = "rv64gc"
RISCV_ABI = "lp64d"
QEMU_CPU = "rv64"
# ==============================================================


YELLOW = "\033[93m"
GREEN = "\033[92m"
RED = "\033[91m"
RESET = "\033[0m"


def log(tag: str, text: str, color: str = YELLOW) -> None:
    print(f"{color}[{tag}]{RESET} {text}", flush=True)


def discover_project() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        project = root / "Compiler" if (root / "Compiler" / "CMakeLists.txt").is_file() else root
        if (project / "CMakeLists.txt").is_file() and (project / "testcases").is_dir():
            return project.resolve()
    raise RuntimeError("未找到项目目录（需要同时包含 CMakeLists.txt 和 testcases/）。")


PROJECT = discover_project()
TEST_ROOT = PROJECT / "testcases"
BUILD_DIR = PROJECT / "build-riscv-release"
COMPILER = BUILD_DIR / "compiler"
WORK_ROOT = PROJECT / ".backendtest-riscv"
CURRENT_DIR = WORK_ROOT / "current"
FAILURE_DIR = WORK_ROOT / "failures"
ARTIFACT_DIR = PROJECT / "backendtest-artifacts"
SYLIB_SOURCE = PROJECT / "runtime" / "sylib.c"
SYLIB_OBJECT = WORK_ROOT / "sylib.o"


def find_toolchain() -> Tuple[str, str]:
    prefixes = []
    configured = os.environ.get("RISCV_PREFIX", "").strip()
    if configured:
        prefixes.append(configured)
    prefixes.extend(("riscv64-linux-gnu-", "riscv64-conda-linux-gnu-"))
    for prefix in prefixes:
        gcc = shutil.which(prefix + "gcc")
        if gcc:
            return gcc, prefix
    raise RuntimeError(
        "未找到 RISC-V GCC；请安装 riscv64-linux-gnu-gcc，"
        "或将工具链前缀写入环境变量 RISCV_PREFIX。"
    )


GCC, GCC_PREFIX = find_toolchain()
QEMU = shutil.which("qemu-riscv64")
if not QEMU:
    raise RuntimeError("未找到 qemu-riscv64，请先安装 qemu-user。")


def first_line(command: list[str]) -> str:
    result = subprocess.run(command, text=True, capture_output=True, check=False)
    return (result.stdout or result.stderr).splitlines()[0]


def prepare_runtime() -> None:
    if WORK_ROOT.exists():
        shutil.rmtree(WORK_ROOT)
    CURRENT_DIR.mkdir(parents=True)
    FAILURE_DIR.mkdir(parents=True)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    if not SYLIB_SOURCE.is_file():
        raise RuntimeError(f"缺少运行库源码：{SYLIB_SOURCE}")
    log("SETUP", "编译 RISC-V 运行库 ...")
    command = [
        GCC, f"-march={RISCV_MARCH}", f"-mabi={RISCV_ABI}",
        "-O2", "-w", "-c", str(SYLIB_SOURCE), "-o", str(SYLIB_OBJECT),
    ]
    result = subprocess.run(
        command, text=True, capture_output=True,
        timeout=ASSEMBLE_TIMEOUT, check=False,
    )
    if result.returncode != 0:
        detail = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"RISC-V 运行库编译失败：\n{detail}")


def collect_cases() -> List[Path]:
    if SINGLE_CASES:
        cases = [(TEST_ROOT / name).resolve() for name in SINGLE_CASES]
        missing = [case for case in cases if not case.is_file()]
        if missing:
            raise RuntimeError("指定样例不存在：" + ", ".join(map(str, missing)))
    else:
        cases = sorted(TEST_ROOT.rglob("*.sy"))
    return cases[:MAX_CASES] if MAX_CASES is not None else cases


def is_performance_case(case: Path) -> bool:
    return any("performance" in part.lower() for part in case.relative_to(TEST_ROOT).parts)


def expected_output(case: Path) -> Path:
    return case.with_suffix(".out")


def stdin_file(case: Path) -> Optional[Path]:
    path = case.with_suffix(".in")
    return path if path.is_file() else None


def run_command(command: List[str], timeout: int, stdin: Optional[Path] = None):
    input_handle = stdin.open("rb") if stdin else subprocess.DEVNULL
    started = time.perf_counter()
    try:
        result = subprocess.run(
            command,
            stdin=input_handle,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
            check=False,
        )
        return result, time.perf_counter() - started, None
    except subprocess.TimeoutExpired as exc:
        return exc, time.perf_counter() - started, "timeout"
    finally:
        if stdin:
            input_handle.close()


def clean_text(data: bytes) -> str:
    """Decode diagnostics; trailing blank lines are unimportant here."""
    return data.decode("utf-8", errors="replace").replace("\r\n", "\n").rstrip("\n")


def normalize_program_output(data: bytes) -> str:
    """Normalize line endings without deleting meaningful trailing newlines."""
    return data.decode("utf-8", errors="replace").replace("\r\n", "\n")


def parse_runtime_seconds(stderr: bytes) -> Optional[float]:
    """Read only timing emitted by runtime/sylib.c, never host pipeline time."""
    timer_pattern = rb"Timer@\d+-\d+:\s*(\d+)H-(\d+)M-(\d+)S-(\d+)us"
    timers = re.findall(timer_pattern, stderr)
    if timers:
        # Sum Timer records as integer microseconds. This also avoids sylib.c's
        # missing carry propagation when TOTAL combines multiple timer regions.
        total_microseconds = sum(
            ((int(h) * 60 + int(m)) * 60 + int(s)) * 1_000_000 + int(us)
            for h, m, s, us in timers
        )
        return total_microseconds / 1_000_000
    total = re.search(rb"TOTAL:\s*(\d+)H-(\d+)M-(\d+)S-(\d+)us", stderr)
    if not total:
        return None
    hours, minutes, seconds, microseconds = map(int, total.groups())
    return hours * 3600 + minutes * 60 + seconds + microseconds / 1_000_000


def format_filename(path: Path, max_length: int = 25) -> str:
    name = path.name
    if len(name) > max_length:
        name = name[: max_length - 3] + "..."
    return name.ljust(max_length)


def print_case(
    status: str, case: Path, runtime: float, total_runtime: float, index: int, total: int
) -> None:
    color = GREEN if status == "AC" else YELLOW if status == "TLE" else RED
    gap = " " if len(status) >= 3 else "  "
    print(
        f"{color}[{status}]{RESET}{gap}{format_filename(case)} | "
        f"Runtime:{runtime:.6f}s | Total:{total_runtime:>7.2f}s | [{index}/{total}]",
        flush=True,
    )


def save_failure(index: int, case: Path, status: str, detail: str) -> None:
    safe_name = re.sub(r"[^0-9A-Za-z_.-]+", "_", str(case.relative_to(TEST_ROOT)))
    destination = FAILURE_DIR / f"{index:03d}_{status}_{safe_name}"
    destination.mkdir(parents=True, exist_ok=True)
    for artifact in CURRENT_DIR.iterdir():
        if artifact.is_file():
            shutil.copy2(artifact, destination / artifact.name)
    (destination / "detail.txt").write_text(detail, encoding="utf-8")


def run_case(
    index: int, total: int, case: Path, total_runtime: float
) -> Tuple[str, float, float, str]:
    """Return status, displayed runtime, amount added to Total, and detail."""
    relative = case.relative_to(TEST_ROOT)
    optimize = is_performance_case(case)
    mode = "-O1" if optimize else "FUNC"
    log("RUN", f"{relative} | {mode} | Total:{total_runtime:>7.2f}s | [{index}/{total}]")

    for artifact in CURRENT_DIR.iterdir():
        if artifact.is_file():
            artifact.unlink()
    source = CURRENT_DIR / "testcase.sysy"
    case_artifact_dir = ARTIFACT_DIR / relative.parent
    case_artifact_dir.mkdir(parents=True, exist_ok=True)
    assembly = case_artifact_dir / f"{case.stem}.s"
    llvm_ir = case_artifact_dir / f"{case.stem}.ll"
    llvm_ir.unlink(missing_ok=True)
    assembly.unlink(missing_ok=True)
    object_file = CURRENT_DIR / "testcase.o"
    executable = CURRENT_DIR / "testcase"
    staged_input = CURRENT_DIR / "testcase.in"
    staged_expected = CURRENT_DIR / "testcase.out"
    shutil.copy2(case, source)
    original_input = stdin_file(case)
    if original_input:
        shutil.copy2(original_input, staged_input)
    original_expected = expected_output(case)
    if original_expected.is_file():
        shutil.copy2(original_expected, staged_expected)

    # compiler 会在 assembly 同目录自动生成同名 .ll 文件。
    compiler_command = [str(COMPILER), str(source), "-S", "-o", str(assembly)]
    if optimize:
        compiler_command.append("-O1")
    compiled, elapsed, error = run_command(compiler_command, COMPILE_TIMEOUT)
    if error:
        detail = f"编译器超时 ({elapsed:.2f}s)\ncommand: {' '.join(compiler_command)}"
        save_failure(index, case, "TLE", detail)
        return "TLE", 0.0, float(COMPILE_TIMEOUT), detail
    if compiled.returncode != 0 or not assembly.is_file():
        detail = clean_text(compiled.stderr) or clean_text(compiled.stdout) or "编译器未生成汇编文件"
        save_failure(index, case, "CE", detail)
        return "CE", 0.0, 0.0, detail
    if not llvm_ir.is_file():
        detail = f"编译器未生成中端 IR 文件：{llvm_ir}"
        save_failure(index, case, "CE", detail)
        return "CE", 0.0, 0.0, detail

    assemble_command = [
        GCC, f"-march={RISCV_MARCH}", f"-mabi={RISCV_ABI}",
        "-c", str(assembly), "-o", str(object_file),
    ]
    assembled, elapsed, error = run_command(assemble_command, ASSEMBLE_TIMEOUT)
    if error:
        detail = f"汇编超时 ({elapsed:.2f}s)"
        save_failure(index, case, "TLE", detail)
        return "TLE", 0.0, float(ASSEMBLE_TIMEOUT), detail
    if assembled.returncode != 0:
        detail = clean_text(assembled.stderr) or clean_text(assembled.stdout)
        save_failure(index, case, "AE", detail)
        return "AE", 0.0, 0.0, detail

    link_command = [
        GCC, f"-march={RISCV_MARCH}", f"-mabi={RISCV_ABI}", "-static",
        str(object_file), str(SYLIB_OBJECT), "-lm", "-o", str(executable),
    ]
    linked, elapsed, error = run_command(link_command, LINK_TIMEOUT)
    if error:
        detail = f"链接超时 ({elapsed:.2f}s)"
        save_failure(index, case, "TLE", detail)
        return "TLE", 0.0, float(LINK_TIMEOUT), detail
    if linked.returncode != 0:
        detail = clean_text(linked.stderr) or clean_text(linked.stdout)
        save_failure(index, case, "LE", detail)
        return "LE", 0.0, 0.0, detail

    qemu_command = [QEMU, "-cpu", QEMU_CPU, str(executable)]
    executed, wall_time, error = run_command(
        qemu_command, RUN_TIMEOUT, staged_input if staged_input.is_file() else None
    )
    if error:
        detail = f"QEMU 运行超时 ({wall_time:.2f}s)"
        save_failure(index, case, "TLE", detail)
        return "TLE", float(RUN_TIMEOUT), float(RUN_TIMEOUT), detail
    if executed.returncode < 0:
        detail = f"QEMU 进程被信号 {-executed.returncode} 终止\n{clean_text(executed.stderr)}"
        save_failure(index, case, "RE", detail)
        return "RE", 0.0, 0.0, detail

    runtime = parse_runtime_seconds(executed.stderr)
    if runtime is None:
        detail = (
            "运行库 stderr 中缺少 TOTAL 计时行；拒绝用主机墙钟时间冒充目标程序时间。\n"
            + clean_text(executed.stderr)
        )
        save_failure(index, case, "BAD", detail)
        return "BAD", 0.0, 0.0, detail
    if not staged_expected.is_file():
        detail = f"缺少期望输出：{original_expected}"
        save_failure(index, case, "BAD", detail)
        return "BAD", runtime, runtime, detail

    # Preserve every program-produced trailing newline. 37_dct.sy intentionally
    # prints a blank line before the harness appends main's return value.
    stdout = normalize_program_output(executed.stdout)
    if stdout and not stdout.endswith("\n"):
        stdout += "\n"
    actual = stdout + f"{executed.returncode}\n"
    expected = staged_expected.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n")
    if actual.splitlines() != expected.splitlines():
        diff = "\n".join(unified_diff(
            expected.splitlines(), actual.splitlines(),
            fromfile="expected", tofile="actual", lineterm="",
        ))
        detail = diff or f"expected={expected!r}\nactual={actual!r}"
        save_failure(index, case, "WA", detail)
        return "WA", runtime, runtime, detail
    return "AC", runtime, runtime, ""


def run_backend_tests() -> Dict[str, List[Path]]:
    gcc_version = first_line([GCC, "--version"])
    qemu_version = first_line([QEMU, "--version"])
    log("ENV", "arch=riscv64, mode=qemu-user")
    log("ENV", f"gcc={gcc_version}")
    log("ENV", f"assembler/link flags: -march={RISCV_MARCH} -mabi={RISCV_ABI}")
    log("ENV", f"qemu={qemu_version}, cpu={QEMU_CPU}")
    log("ENV", "SIMD/vector: disabled by -march=rv64gc (no V extension)")
    log("WARN", "QEMU user-mode 只验证 RV64GC/FPU/ABI 与功能，不能模拟 BOOM v3、50MHz、缓存、乱序或 2GB 内存性能。")
    if "13.3" not in gcc_version:
        log("WARN", "本机 GCC 不是竞赛指定的 13.3.0；最终提交前必须使用指定版本复验。")

    if not COMPILER.is_file():
        raise RuntimeError(
            f"未找到已构建的 RISC-V Release 编译器：{COMPILER}\n"
            "请先在 VS Code 中运行 RISC-V 后端 Release 构建任务。"
        )
    log("SETUP", f"使用现有编译器：{COMPILER}")
    prepare_runtime()
    log("SETUP", f"中端 IR/汇编归档：{ARTIFACT_DIR}")

    cases = collect_cases()
    if not cases:
        raise RuntimeError(f"{TEST_ROOT} 下没有找到 .sy 测试样例。")
    log("SETUP", f"cases={len(cases)}, compiler={COMPILER.name}")

    results = {name: [] for name in ("AC", "CE", "AE", "LE", "RE", "WA", "TLE", "BAD")}
    total_runtime = 0.0
    for index, case in enumerate(cases, 1):
        status, runtime, total_increment, detail = run_case(
            index, len(cases), case, total_runtime
        )
        total_runtime += total_increment
        results[status].append(case)
        print_case(status, case, runtime, total_runtime, index, len(cases))
        if status != "AC" and detail:
            log("DETAIL", detail.splitlines()[0], RED)
        if status != "AC" and STOP_ON_FIRST_FAIL:
            break

    print()
    print("=" * 60)
    labels = (
        ("Compiler Error", "CE"), ("Assembler Error", "AE"),
        ("Linker Error", "LE"), ("Runtime Error", "RE"),
        ("Timeout Error", "TLE"), ("Wrong Answer", "WA"),
        ("Bad Test", "BAD"), ("Accepted", "AC"),
    )
    for label, status in labels:
        print(f"{label + ':':<18}{len(results[status]):>3}")
    print("=" * 60)
    print(f"Total time: {total_runtime:.2f}s")
    if any(results[name] for name in results if name != "AC"):
        log("REPORT", f"失败现场：{FAILURE_DIR}", RED)
    return results


results = run_backend_tests()


[ENV] arch=riscv64, mode=qemu-user


[ENV] gcc=riscv64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
[ENV] assembler/link flags: -march=rv64gc -mabi=lp64d
[ENV] qemu=qemu-riscv64 version 6.2.0 (Debian 1:6.2+dfsg-2ubuntu6.30), cpu=rv64
[ENV] SIMD/vector: disabled by -march=rv64gc (no V extension)
[WARN] QEMU user-mode 只验证 RV64GC/FPU/ABI 与功能，不能模拟 BOOM v3、50MHz、缓存、乱序或 2GB 内存性能。
[WARN] 本机 GCC 不是竞赛指定的 13.3.0；最终提交前必须使用指定版本复验。
[SETUP] 使用现有编译器：/home/lyjy/Compiler/Compiler/build-riscv-release/compiler
[SETUP] 编译 RISC-V 运行库 ...
[SETUP] 中端 IR/汇编归档：/home/lyjy/Compiler/Compiler/backendtest-artifacts
[SETUP] cases=382, compiler=compiler
[RUN] final24/h_performance/2024-0ZE-13.sy | -O1 | Total:   0.00s | [1/382]
[AC]  2024-0ZE-13.sy            | Runtime:0.638426s | Total:   0.64s | [1/382]
[RUN] final24/h_performance/2024-1WO-11.sy | -O1 | Total:   0.64s | [2/382]
[AC]  2024-1WO-11.sy            | Runtime:12.251526s | Total:  12.89s | [2/382]
[RUN] final24/h_performance/2024-2KM-21.sy | -O1 | Total:  12.89s | [3/382]
[AC]  2024-2K